# 🍎 Fruits & Vegetables AI – Block 3: Computer Vision (CLIP Classification)

## Objective
Use CLIP (Zero-Shot) to classify fruits and vegetables from images
without any model training or labelled image dataset.

## Approach
Zero-Shot classification with `openai/clip-vit-base-patch32` — the model was never
fine-tuned on our specific classes. It uses its pretrained visual-language knowledge
to match images to text labels.

## Integration with other blocks
- **Output to ML block:** detected food label → look up macronutrients in `nutrition_lookup.json` → predict calories
- **Output to NLP block:** food label + confidence → generate recipe & health advice
- **Quantitative evaluation:** see `cv_evaluation.ipynb` (accuracy, confusion matrix, error analysis)

In [ ]:
from transformers import pipeline
from PIL import Image
import matplotlib.pyplot as plt
import requests
from io import BytesIO
import pandas as pd
import numpy as np
import json

print('Loading CLIP model...')
clip_classifier = pipeline(
    model='openai/clip-vit-base-patch32',
    task='zero-shot-image-classification'
)
print('✅ CLIP loaded!')

## 1. Define Labels

In [ ]:
FRUIT_LABELS = [
    'apple', 'banana', 'orange', 'strawberry', 'grape', 'watermelon',
    'pineapple', 'mango', 'peach', 'pear', 'cherry', 'lemon'
]

VEGETABLE_LABELS = [
    'carrot', 'broccoli', 'tomato', 'cucumber', 'pepper', 'spinach',
    'potato', 'onion', 'garlic', 'lettuce', 'corn', 'mushroom'
]

ALL_LABELS = FRUIT_LABELS + VEGETABLE_LABELS
print(f'Total labels: {len(ALL_LABELS)}')
print('Fruits:', FRUIT_LABELS)
print('Vegetables:', VEGETABLE_LABELS)

## 2. Classification Function

In [ ]:
def classify_food(image_path_or_url):
    """Classify a fruit or vegetable from an image using CLIP Zero-Shot.
    
    Args:
        image_path_or_url: local file path or public URL
    Returns:
        dict with top_label, top_score, top3 predictions, is_fruit flag
    """
    if str(image_path_or_url).startswith('http'):
        response = requests.get(image_path_or_url, timeout=10)
        image = Image.open(BytesIO(response.content)).convert('RGB')
    else:
        image = Image.open(image_path_or_url).convert('RGB')

    results = clip_classifier(image, candidate_labels=ALL_LABELS)
    top3 = results[:3]

    return {
        'top_label': top3[0]['label'],
        'top_score': round(top3[0]['score'], 4),
        'top3': top3,
        'is_fruit': top3[0]['label'] in FRUIT_LABELS
    }

print('✅ classify_food() function ready')

## 3. Test with Example Images

In [ ]:
# Test on 4 public domain images
test_images = {
    'apple':    'https://upload.wikimedia.org/wikipedia/commons/thumb/1/15/Red_Apple.jpg/600px-Red_Apple.jpg',
    'banana':   'https://upload.wikimedia.org/wikipedia/commons/thumb/8/8a/Banana-Fruit-Bundle.jpg/640px-Banana-Fruit-Bundle.jpg',
    'carrot':   'https://upload.wikimedia.org/wikipedia/commons/thumb/a/a2/Vegetable-Carrot-Bundle-wStem.jpg/640px-Vegetable-Carrot-Bundle-wStem.jpg',
    'broccoli': 'https://upload.wikimedia.org/wikipedia/commons/thumb/0/03/Broccoli_and_cross_section_edit.jpg/640px-Broccoli_and_cross_section_edit.jpg'
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
correct = 0

for idx, (true_label, url) in enumerate(test_images.items()):
    response = requests.get(url, timeout=10)
    img    = Image.open(BytesIO(response.content)).convert('RGB')
    result = classify_food(url)
    is_correct = result['top_label'] == true_label
    if is_correct: correct += 1

    axes[idx].imshow(img)
    axes[idx].set_title(
        f'True: {true_label}\nPred: {result["top_label"]}\n({result["top_score"]*100:.1f}%)',
        color='green' if is_correct else 'red', fontsize=10
    )
    axes[idx].axis('off')
    print(f'{true_label}: pred={result["top_label"]} ({result["top_score"]*100:.1f}%) '
          f'{"✅" if is_correct else "❌"}')

plt.suptitle(f'CLIP Zero-Shot Classification – {correct}/4 correct', fontsize=13)
plt.tight_layout()
plt.savefig('../data/cv_results.png', dpi=100)
plt.show()
print(f'\nAccuracy on 4 examples: {correct}/4')

## 4. CV → ML Bridge: Nutrition Lookup

In [ ]:
# Typical macronutrients per 100g for all 24 labels
# This lookup bridges CV output (food label) to ML input (macronutrient features)
NUTRITION_LOOKUP = {
    'apple':       {'protein': 0.3, 'fat': 0.2, 'carbs': 14.0, 'fiber': 2.4},
    'banana':      {'protein': 1.1, 'fat': 0.3, 'carbs': 23.0, 'fiber': 2.6},
    'orange':      {'protein': 0.9, 'fat': 0.1, 'carbs': 12.0, 'fiber': 2.4},
    'strawberry':  {'protein': 0.7, 'fat': 0.3, 'carbs': 8.0,  'fiber': 2.0},
    'grape':       {'protein': 0.6, 'fat': 0.2, 'carbs': 18.0, 'fiber': 0.9},
    'watermelon':  {'protein': 0.6, 'fat': 0.2, 'carbs': 8.0,  'fiber': 0.4},
    'pineapple':   {'protein': 0.5, 'fat': 0.1, 'carbs': 13.0, 'fiber': 1.4},
    'mango':       {'protein': 0.8, 'fat': 0.4, 'carbs': 15.0, 'fiber': 1.6},
    'peach':       {'protein': 0.9, 'fat': 0.3, 'carbs': 10.0, 'fiber': 1.5},
    'pear':        {'protein': 0.4, 'fat': 0.1, 'carbs': 15.0, 'fiber': 3.1},
    'cherry':      {'protein': 1.0, 'fat': 0.3, 'carbs': 16.0, 'fiber': 2.1},
    'lemon':       {'protein': 1.1, 'fat': 0.3, 'carbs': 9.0,  'fiber': 2.8},
    'carrot':      {'protein': 0.9, 'fat': 0.2, 'carbs': 10.0, 'fiber': 2.8},
    'broccoli':    {'protein': 2.8, 'fat': 0.4, 'carbs': 7.0,  'fiber': 2.6},
    'tomato':      {'protein': 0.9, 'fat': 0.2, 'carbs': 3.9,  'fiber': 1.2},
    'cucumber':    {'protein': 0.7, 'fat': 0.1, 'carbs': 3.6,  'fiber': 0.5},
    'pepper':      {'protein': 1.0, 'fat': 0.3, 'carbs': 6.0,  'fiber': 2.1},
    'spinach':     {'protein': 2.9, 'fat': 0.4, 'carbs': 3.6,  'fiber': 2.2},
    'potato':      {'protein': 2.0, 'fat': 0.1, 'carbs': 17.0, 'fiber': 2.2},
    'onion':       {'protein': 1.1, 'fat': 0.1, 'carbs': 9.3,  'fiber': 1.7},
    'garlic':      {'protein': 6.4, 'fat': 0.5, 'carbs': 33.0, 'fiber': 2.1},
    'lettuce':     {'protein': 1.4, 'fat': 0.2, 'carbs': 2.9,  'fiber': 1.3},
    'corn':        {'protein': 3.3, 'fat': 1.5, 'carbs': 19.0, 'fiber': 2.7},
    'mushroom':    {'protein': 3.1, 'fat': 0.3, 'carbs': 3.3,  'fiber': 1.0},
}

# Save for use in app
with open('../data/nutrition_lookup.json', 'w') as f:
    json.dump(NUTRITION_LOOKUP, f, indent=2)

print(f'✅ nutrition_lookup.json saved ({len(NUTRITION_LOOKUP)} entries)')
print('\nExample – apple lookup:')
print(f'  {NUTRITION_LOOKUP["apple"]}')
print('\nThis bridges CV → ML: CLIP detects "apple" → lookup macros → Ridge predicts calories')

## 5. Results Summary

### Qualitative Results (4 example images)

| Image | True Label | Predicted | Confidence | Correct? |
| --- | --- | --- | --- | --- |
| Red apple | apple | apple | ~95% | ✅ |
| Banana bunch | banana | banana | ~98% | ✅ |
| Carrot bundle | carrot | carrot | ~92% | ✅ |
| Broccoli | broccoli | broccoli | ~97% | ✅ |

### Quantitative Evaluation
Full quantitative evaluation (accuracy, confusion matrix, error analysis) is documented
in **`cv_evaluation.ipynb`**:
- **Overall Top-1 Accuracy: 81.8%** on 209 test images (18 classes)
- **Avg. confidence (correct): 94.0%** vs **incorrect: 65.2%**
- **Main failure:** pepper → tomato (20/30 images) due to visual colour/shape overlap

### Why Zero-Shot?
Zero-Shot classification was chosen because:
1. No labelled training images needed — reduces project complexity
2. CLIP generalises well to new images without fine-tuning
3. 81.8% accuracy without any training is strong for a production app
4. New food categories can be added by simply adding a label string

### Limitations
- Pepper class fails completely (F1=0.00) — visually similar to tomato and cucumber
- Confidence threshold (e.g. < 70%) could be used to warn users about uncertain predictions
- Fine-tuning on a dedicated dataset would push accuracy above 95%